# 🎵 TikTok AI Agent
**Full-featured TikTok automation with human-like behavior using Selenium.**

### Features:
- 🔐 Login (cookie session persistence)
- 📤 Upload Videos
- ❤️ Like Videos
- 💬 Comment & Reply
- 👥 Follow / Unfollow Users
- 📊 View Video Analytics
- 🔍 Search by Hashtag / User
- 📩 Send Messages
- ⏰ Post Scheduler

> ⚠️ Uses `undetected-chromedriver` + realistic mouse movements & random delays.

In [ ]:
# ============================================================
# CELL 1: Imports & Human Behavior Engine
# ============================================================
import time
import random
import pickle
import os
import threading
from pathlib import Path
from datetime import datetime
from getpass import getpass

import undetected_chromedriver as uc
from selenium.webdriver.common.by import By
from selenium.webdriver.common.keys import Keys
from selenium.webdriver.common.action_chains import ActionChains
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.common.exceptions import TimeoutException, NoSuchElementException

import colorama
from colorama import Fore, Style
colorama.init()

# ── Human behavior helpers ───────────────────────────────────
def human_delay(mn=1.5, mx=4.5):
    t = random.uniform(mn, mx)
    time.sleep(t)

def long_delay(mn=8.0, mx=25.0):
    t = random.uniform(mn, mx)
    print(f"{Fore.YELLOW}  😴 pause {t:.1f}s...{Style.RESET_ALL}")
    time.sleep(t)

def scroll_down(driver, pixels: int = None):
    px = pixels or random.randint(300, 900)
    driver.execute_script(f"window.scrollBy(0, {px})")
    time.sleep(random.uniform(0.5, 1.5))

def human_type(element, text: str):
    element.clear()
    time.sleep(random.uniform(0.2, 0.5))
    for ch in text:
        element.send_keys(ch)
        time.sleep(random.uniform(0.05, 0.14))
    time.sleep(random.uniform(0.3, 0.7))

def safe_click(driver, el):
    driver.execute_script("arguments[0].scrollIntoView({block:'center'})", el)
    time.sleep(random.uniform(0.3, 0.7))
    ActionChains(driver).move_to_element(el).pause(random.uniform(0.1, 0.4)).click().perform()

def wait(driver, css, timeout=15):
    return WebDriverWait(driver, timeout).until(
        EC.presence_of_element_located((By.CSS_SELECTOR, css))
    )

def wait_click(driver, css, timeout=15):
    return WebDriverWait(driver, timeout).until(
        EC.element_to_be_clickable((By.CSS_SELECTOR, css))
    )

def log(msg, level="info"):
    c = {"info": Fore.GREEN, "warn": Fore.YELLOW, "error": Fore.RED, "data": Fore.MAGENTA}.get(level, Fore.WHITE)
    print(f"{c}[{datetime.now():%H:%M:%S}] {msg}{Style.RESET_ALL}")

def simulate_watch(driver, seconds: float = None):
    """Simulate watching a video for a natural duration."""
    watch_time = seconds or random.uniform(4, 25)
    log(f"  ▶️  Watching for {watch_time:.1f}s...")
    time.sleep(watch_time)

log("✅ Imports & human engine loaded.")

In [ ]:
# ============================================================
# CELL 2: Browser Setup & TikTok Login
# ============================================================
USERNAME     = input("TikTok username (or email): ")
PASSWORD     = getpass("TikTok password: ")
COOKIES_FILE = Path(f"tiktok_cookies_{USERNAME.replace('@','')}.pkl")

def create_driver():
    opts = uc.ChromeOptions()
    opts.add_argument("--start-maximized")
    opts.add_argument("--disable-blink-features=AutomationControlled")
    opts.add_argument("--lang=en-US")
    driver = uc.Chrome(options=opts)
    # Mask WebDriver presence
    driver.execute_cdp_cmd("Page.addScriptToEvaluateOnNewDocument", {
        "source": """
            Object.defineProperty(navigator, 'webdriver', {get: () => undefined});
            Object.defineProperty(navigator, 'plugins', {get: () => [1,2,3]});
        """
    })
    return driver

def tiktok_login(driver):
    driver.get("https://www.tiktok.com/")
    human_delay(3, 5)

    # Load cookies if available
    if COOKIES_FILE.exists():
        log("Loading saved session cookies...")
        for c in pickle.loads(COOKIES_FILE.read_bytes()):
            try: driver.add_cookie(c)
            except: pass
        driver.refresh()
        human_delay(4, 7)
        if driver.find_elements(By.CSS_SELECTOR, "[data-e2e='profile-icon']"):
            log("✅ Logged in via saved session!")
            return

    # Manual login via email
    driver.get("https://www.tiktok.com/login/phone-or-email/email")
    human_delay(3, 5)

    email_el = wait(driver, "input[name='username']")
    human_type(email_el, USERNAME)
    human_delay(1, 2)

    pass_el = wait(driver, "input[type='password']")
    human_type(pass_el, PASSWORD)
    human_delay(1, 2)

    login_btn = wait_click(driver, "button[type='submit']")
    safe_click(driver, login_btn)
    human_delay(5, 10)

    # Handle CAPTCHA manually if needed
    if "captcha" in driver.current_url or "verify" in driver.page_source.lower():
        log("⚠️  CAPTCHA detected — please solve it in the browser then press Enter here.", "warn")
        input("Press Enter after solving CAPTCHA...")
        human_delay(2, 4)

    COOKIES_FILE.write_bytes(pickle.dumps(driver.get_cookies()))
    log("✅ Logged in & cookies saved.")

driver = create_driver()
tiktok_login(driver)
log(f"TikTok session active for @{USERNAME}.", "data")

In [ ]:
# ============================================================
# CELL 3: Upload a Video
# ============================================================
def upload_video(video_path: str, caption: str, hashtags: list = None):
    """
    Upload a video to TikTok.
    video_path : absolute path to .mp4 file
    caption    : post description
    hashtags   : list of hashtags without # e.g. ['fyp','viral']
    """
    log(f"Uploading video: {video_path}")
    driver.get("https://www.tiktok.com/upload?lang=en")
    human_delay(4, 7)

    # Upload file
    file_input = wait(driver, "input[type='file']")
    file_input.send_keys(os.path.abspath(video_path))
    log("  📤 File selected, waiting for upload...")
    long_delay(10, 25)   # wait for upload + processing

    # Write caption
    caption_box = wait(driver, ".public-DraftEditor-content, [contenteditable='true']")
    safe_click(driver, caption_box)

    full_caption = caption
    if hashtags:
        full_caption += " " + " ".join([f"#{h}" for h in hashtags])
    human_type(caption_box, full_caption)
    human_delay(2, 4)

    # Click Post
    post_btn = wait_click(driver, "button.btn-post")
    safe_click(driver, post_btn)
    human_delay(5, 10)
    log("✅ Video posted!", "data")
    long_delay(15, 30)

# ── Example ──────────────────────────────────────────────────
# upload_video(
#     r"C:\videos\myclip.mp4",
#     caption="Check this out! 🔥",
#     hashtags=["fyp", "viral", "trending"]
# )

In [ ]:
# ============================================================
# CELL 4: Browse Feed & Like Videos
# ============================================================
def browse_and_like(count: int = 5, watch_avg: float = 10.0):
    """
    Simulate a real user browsing FYP and liking videos.
    count     : how many videos to interact with
    watch_avg : average watch time per video in seconds
    """
    log(f"Browsing FYP, will like ~{count} videos...")
    driver.get("https://www.tiktok.com/foryou")
    human_delay(4, 7)

    liked = 0
    for i in range(count * 2):
        if liked >= count:
            break

        # Watch video naturally
        watch_time = random.gauss(watch_avg, 5)
        simulate_watch(driver, max(3, watch_time))

        # Like with 70% probability
        if random.random() < 0.70:
            try:
                like_btn = driver.find_element(By.CSS_SELECTOR, "[data-e2e='like-icon']")
                safe_click(driver, like_btn)
                liked += 1
                log(f"  ❤️  Liked video #{liked}")
                human_delay(1, 3)
            except Exception as e:
                log(f"  Like failed: {e}", "warn")

        # Scroll to next video
        driver.find_element(By.TAG_NAME, "body").send_keys(Keys.ARROW_DOWN)
        human_delay(1, 3)

    log(f"✅ Liked {liked} videos.", "data")

def like_video_by_url(video_url: str):
    """Navigate to a specific video and like it."""
    driver.get(video_url)
    human_delay(3, 6)
    simulate_watch(driver, random.uniform(5, 15))
    like_btn = wait(driver, "[data-e2e='like-icon']")
    safe_click(driver, like_btn)
    log("✅ Video liked.", "data")
    human_delay(2, 5)

# ── Example ──────────────────────────────────────────────────
# browse_and_like(count=5, watch_avg=12)

In [ ]:
# ============================================================
# CELL 5: Comment & Reply
# ============================================================
def comment_on_video(video_url: str, comment_text: str):
    """Watch a video and leave a comment."""
    log(f"Commenting on video: {video_url}")
    driver.get(video_url)
    human_delay(3, 6)
    simulate_watch(driver, random.uniform(5, 12))

    # Open comment section
    try:
        comment_icon = wait(driver, "[data-e2e='comment-icon'], .comment-enter")
        safe_click(driver, comment_icon)
        human_delay(1, 3)
    except:
        pass   # comments may already be visible

    # Type comment
    comment_box = wait(driver, "[data-e2e='comment-input'], .CommentInput")
    safe_click(driver, comment_box)
    human_type(comment_box, comment_text)
    human_delay(1, 2)

    # Submit
    comment_box.send_keys(Keys.RETURN)
    human_delay(2, 5)
    log(f"✅ Commented: '{comment_text}'", "data")
    long_delay(8, 20)

def reply_to_comment_on_video(video_url: str, reply_text: str, reply_to_user: str = None):
    """Reply to a comment on a video. If reply_to_user is given, finds their comment."""
    driver.get(video_url)
    human_delay(3, 6)
    simulate_watch(driver, random.uniform(4, 10))

    comments = driver.find_elements(By.CSS_SELECTOR, ".comment-item, [data-e2e='comment-level-1']")
    target = None
    for c in comments:
        try:
            uname = c.find_element(By.CSS_SELECTOR, "[data-e2e='comment-username-1']").text
            if not reply_to_user or reply_to_user.lower() in uname.lower():
                target = c
                break
        except: continue

    if not target:
        log("Target comment not found.", "warn")
        return

    reply_btn = target.find_element(By.CSS_SELECTOR, "[data-e2e='comment-reply-btn']")
    safe_click(driver, reply_btn)
    human_delay(1, 3)

    reply_box = wait(driver, "[data-e2e='comment-input']")
    human_type(reply_box, reply_text)
    reply_box.send_keys(Keys.RETURN)
    log(f"✅ Reply posted.", "data")
    long_delay(8, 18)

# ── Example ──────────────────────────────────────────────────
# comment_on_video("https://www.tiktok.com/@user/video/1234567890", "This is amazing 🔥")

In [ ]:
# ============================================================
# CELL 6: Follow & Unfollow Users
# ============================================================
def follow_user(username: str):
    """Follow a TikTok user."""
    log(f"Following @{username}...")
    driver.get(f"https://www.tiktok.com/@{username}")
    human_delay(3, 6)

    try:
        follow_btn = wait(driver, "[data-e2e='follow-button']")
        btn_text   = follow_btn.text.lower()
        if "follow" in btn_text and "following" not in btn_text:
            safe_click(driver, follow_btn)
            log(f"✅ Followed @{username}.", "data")
        else:
            log(f"Already following @{username}.", "warn")
    except Exception as e:
        log(f"Follow failed: {e}", "warn")

    long_delay(5, 15)

def unfollow_user(username: str):
    """Unfollow a TikTok user."""
    log(f"Unfollowing @{username}...")
    driver.get(f"https://www.tiktok.com/@{username}")
    human_delay(3, 6)

    try:
        follow_btn = wait(driver, "[data-e2e='follow-button']")
        if "following" in follow_btn.text.lower():
            safe_click(driver, follow_btn)
            human_delay(1, 2)
            # Confirm unfollow dialog if it appears
            try:
                confirm = wait_click(driver, "button.confirm-btn", timeout=4)
                safe_click(driver, confirm)
            except: pass
            log(f"✅ Unfollowed @{username}.", "data")
    except Exception as e:
        log(f"Unfollow failed: {e}", "warn")

    long_delay(5, 15)

# ── Example ──────────────────────────────────────────────────
# follow_user("charlidamelio")

In [ ]:
# ============================================================
# CELL 7: View Analytics on Your Videos
# ============================================================
def get_my_video_analytics():
    """Navigate to creator studio to see video analytics."""
    log("Opening Creator Studio analytics...")
    driver.get("https://www.tiktok.com/creator-center/analytics")
    human_delay(5, 8)
    log("📊 Analytics page opened in browser. Review the dashboard.", "data")

def get_profile_stats(username: str):
    """Scrape basic stats from a TikTok profile."""
    log(f"Getting stats for @{username}...")
    driver.get(f"https://www.tiktok.com/@{username}")
    human_delay(3, 6)

    try:
        following = driver.find_element(By.CSS_SELECTOR, "[data-e2e='following-count']").text
        followers = driver.find_element(By.CSS_SELECTOR, "[data-e2e='followers-count']").text
        likes     = driver.find_element(By.CSS_SELECTOR, "[data-e2e='likes-count']").text

        print("\n" + "═" * 40)
        print(f"  📊 @{username} Stats")
        print("═" * 40)
        print(f"  👥 Following : {following}")
        print(f"  🧑‍🤝‍🧑 Followers : {followers}")
        print(f"  ❤️  Total Likes: {likes}")
        print("═" * 40 + "\n")
        return {"following": following, "followers": followers, "likes": likes}
    except Exception as e:
        log(f"Could not fetch stats: {e}", "warn")

# ── Example ──────────────────────────────────────────────────
# get_profile_stats(USERNAME)
# get_my_video_analytics()

In [ ]:
# ============================================================
# CELL 8: Search by Hashtag
# ============================================================
def explore_hashtag(hashtag: str, like_count: int = 3):
    """
    Explore a TikTok hashtag page, watch and like videos.
    """
    log(f"Exploring #{hashtag}...")
    driver.get(f"https://www.tiktok.com/tag/{hashtag}")
    human_delay(3, 6)

    # Click on first video to start watching
    try:
        first_video = wait(driver, "[data-e2e='challenge-item'], .video-feed-item")
        safe_click(driver, first_video)
        human_delay(2, 4)
    except Exception as e:
        log(f"Could not open video: {e}", "warn")
        return

    liked = 0
    for _ in range(like_count * 2):
        if liked >= like_count:
            break
        simulate_watch(driver, random.uniform(5, 18))
        if random.random() < 0.7:
            try:
                like_btn = driver.find_element(By.CSS_SELECTOR, "[data-e2e='like-icon']")
                safe_click(driver, like_btn)
                liked += 1
                log(f"  ❤️  Liked video #{liked} from #{hashtag}")
            except: pass
        driver.find_element(By.TAG_NAME, "body").send_keys(Keys.ARROW_DOWN)
        human_delay(1, 3)

    log(f"✅ Done with #{hashtag}. Liked {liked} videos.", "data")

# ── Example ──────────────────────────────────────────────────
# explore_hashtag("python", like_count=4)

In [ ]:
# ============================================================
# CELL 9: Send Direct Messages
# ============================================================
def send_dm(username: str, message: str):
    """Send a DM to a TikTok user."""
    log(f"Sending DM to @{username}...")
    driver.get(f"https://www.tiktok.com/@{username}")
    human_delay(3, 6)

    try:
        msg_btn = wait(driver, "[data-e2e='message-btn']")
        safe_click(driver, msg_btn)
        human_delay(2, 4)

        msg_input = wait(driver, "[data-e2e='message-input'], .message-input")
        safe_click(driver, msg_input)
        human_type(msg_input, message)
        human_delay(1, 2)
        msg_input.send_keys(Keys.RETURN)

        log(f"✅ DM sent to @{username}.", "data")
    except Exception as e:
        log(f"DM failed: {e}", "warn")

    long_delay(10, 25)

# ── Example ──────────────────────────────────────────────────
# send_dm("someuser", "Hey! Loved your video 🔥")

In [ ]:
# ============================================================
# CELL 10: Post Scheduler
# ============================================================
import schedule as sched

def schedule_video_upload(video_path: str, caption: str, hashtags: list, hour: int, minute: int):
    time_str = f"{hour:02d}:{minute:02d}"
    def job():
        log(f"⏰ Uploading scheduled video at {time_str}")
        upload_video(video_path, caption, hashtags)
    sched.every().day.at(time_str).do(job)
    log(f"📅 Video upload scheduled for {time_str}.")

def run_scheduler():
    def _loop():
        while True:
            sched.run_pending()
            time.sleep(30)
    threading.Thread(target=_loop, daemon=True).start()
    log("✅ Scheduler running in background.")

# ── Example ──────────────────────────────────────────────────
# schedule_video_upload(r"C:\videos\myclip.mp4", "Daily vlog! 🎬", ["vlog","fyp"], hour=18, minute=0)
# run_scheduler()

In [ ]:
# ============================================================
# CELL 11: Daily Routine (One-Click)
# ============================================================
def daily_routine(hashtags: list, like_per_tag: int = 3):
    """
    Simulate a human daily TikTok session:
     1. Browse FYP and like videos
     2. Explore hashtags
    """
    log("🌅 Starting daily TikTok routine...")

    # Browse FYP
    log("Step 1: Browsing FYP...")
    try:
        browse_and_like(count=5, watch_avg=12)
    except Exception as e:
        log(f"FYP error: {e}", "warn")
    long_delay(15, 30)

    # Explore hashtags
    log("Step 2: Exploring hashtags...")
    for tag in hashtags:
        try:
            explore_hashtag(tag, like_count=like_per_tag)
        except Exception as e:
            log(f"#{tag} error: {e}", "warn")
        long_delay(20, 40)

    log("✅ Daily TikTok routine complete!", "data")

# ── Example ──────────────────────────────────────────────────
# daily_routine(["fyp", "comedy", "trending"], like_per_tag=3)

In [ ]:
# ============================================================
# CELL 12: Cleanup
# ============================================================
def close_browser():
    log("Closing browser...")
    driver.quit()
    log("✅ Done.")

# close_browser()